# 01 — Unduh Dataset ID_Regulations

**Dataset:** `horelulus/ID_Regulations` (Hugging Face) — >250.000 PDF peraturan Indonesia, lisensi **Apache-2.0**.

## Struktur dataset yang sebenarnya

Repo **tidak** menyimpan PDF lepas maupun file parquet. Isinya:

- **293 berkas ZIP** bernama `batch-<N>-chunk-<M>.zip`.
- Ukuran setiap ZIP **bervariasi** (mulai ±90 MB; sebagian >2 GB) dan berisi **ratusan–ribuan PDF**.

| Mode | Jumlah ZIP | Perkiraan PDF | Perkiraan unduhan | Perkiraan disk |
|---|---|---|---|---|
| `subset` (default) | 1 | ratusan | mulai ±90 MB | hingga beberapa GB |
| `full` | 293 | ±250.000 | puluhan GB | puluhan GB |

> **Catatan:** dataset & hasil ekstraksi disimpan di `data/raw/`, yang **di-*ignore* Git**
> (lihat `.gitignore`). Jangan pernah `git add` isi `data/raw/`.

**Prasyarat:** `pip install huggingface_hub` (sudah termasuk dependensi `datasets`).

## 0. Konfigurasi

- `MODE = "subset"` → ambil `MAX_ZIPS` arsip pertama (aman untuk uji pipeline).
- `MODE = "full"` → unduh seluruh arsip (butuh puluhan GB; jalankan hanya bila disk & koneksi siap).
- `MAX_DOCS > 0` → batasi jumlah PDF yang diekstrak (berguna untuk uji cepat).

In [ ]:
import hashlib
import json
import re
import zipfile
from datetime import date
from pathlib import Path

REPO_ID = "horelulus/ID_Regulations"
BASE_DIR = Path("data/raw/ID_Regulations")   # sesuai .gitignore (data/raw/*)
ZIP_DIR = BASE_DIR / "_zips"
PDF_DIR = BASE_DIR / "pdfs"

MODE = "subset"       # "subset" atau "full"
MAX_ZIPS = 1          # dipakai bila MODE == "subset" (jumlah arsip; ukuran tiap arsip bervariasi)
MAX_DOCS = 0          # 0 = tanpa batas; isi mis. 1000 untuk uji cepat

for d in (BASE_DIR, ZIP_DIR, PDF_DIR):
    d.mkdir(parents=True, exist_ok=True)

from huggingface_hub import HfApi

api = HfApi()
all_files = api.list_repo_files(REPO_ID, repo_type="dataset")
def _zip_key(name):
    # urutkan berdasarkan angka batch lalu chunk (bukan alfabetis)
    m = re.search(r"batch-(\d+)-chunk-(\d+)", name)
    return (int(m.group(1)), int(m.group(2))) if m else (10**9, 0)

zips = sorted((f for f in all_files if f.lower().endswith(".zip")), key=_zip_key)

print("Arsip ZIP di repo :", len(zips))
print("Mode              :", MODE, "| MAX_ZIPS =", MAX_ZIPS, "| MAX_DOCS =", MAX_DOCS)
print("Tujuan            :", BASE_DIR.resolve())

## 1. Unduh ZIP lalu ekstrak PDF

Nama berkas PDF diberi awalan nama ZIP agar tidak bertabrakan antar-arsip
(contoh: `batch-1-chunk-1__12345.pdf`).

> **Butuh waktu.** Mengunduh satu ZIP (ukurannya bervariasi) dan mengekstrak isinya bisa memakan
> beberapa menit. Untuk `full`, proses bisa berjam-jam.

In [ ]:
from huggingface_hub import hf_hub_download

chosen = zips if MODE == "full" else zips[:MAX_ZIPS]

extracted = 0
zip_records = []

for i, fname in enumerate(chosen, 1):
    local_zip = hf_hub_download(
        repo_id=REPO_ID, filename=fname, repo_type="dataset", local_dir=str(ZIP_DIR)
    )
    zip_path = Path(local_zip)

    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith(".pdf")]
        for m in members:
            if MAX_DOCS and extracted >= MAX_DOCS:
                break
            out_name = f"{zip_path.stem}__{Path(m).name}"
            target = PDF_DIR / out_name
            if not target.exists():
                with zf.open(m) as src, open(target, "wb") as dst:
                    dst.write(src.read())
            extracted += 1

    zip_records.append({
        "zip": fname,
        "zip_bytes": zip_path.stat().st_size,
        "pdf_in_zip": len(members),
    })
    print(f"[{i}/{len(chosen)}] {fname}: {len(members)} PDF (total diekstrak: {extracted})")

    if MAX_DOCS and extracted >= MAX_DOCS:
        print("Batas MAX_DOCS tercapai, berhenti.")
        break

print("Total PDF diekstrak:", extracted)

## 2. Verifikasi jumlah & ukuran

In [ ]:
pdfs = sorted(PDF_DIR.glob("*.pdf"))
total_bytes = sum(f.stat().st_size for f in pdfs)

print("Jumlah berkas PDF :", len(pdfs))
print("Total ukuran      :", round(total_bytes / 1024 / 1024, 2), "MB")
print("Lokasi            :", PDF_DIR.resolve())
print("Contoh nama       :", [p.name for p in pdfs[:5]])

## 3. Manifest & hash (untuk Tabel 1.4)

Hash dihitung pada **berkas ZIP** (lebih cepat dibanding menghitung ribuan PDF),
sekaligus mencatat jumlah PDF yang diekstrak.

In [ ]:
manifest = {
    "dataset": REPO_ID,
    "license": "Apache-2.0",
    "downloaded": date.today().isoformat(),
    "mode": MODE,
    "count_pdf": len(pdfs),
    "total_pdf_bytes": total_bytes,
    "zips": [
        {
            **rec,
            "sha256": hashlib.sha256((ZIP_DIR / rec["zip"]).read_bytes()).hexdigest(),
        }
        for rec in zip_records
    ],
}

out = BASE_DIR / "_manifest.json"
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print("Manifest ditulis:", out.resolve())

# --- nilai untuk tabel 1.4 di README ---
print("Jumlah dokumen  :", manifest["count_pdf"])
print("Tanggal unduh   :", manifest["downloaded"])
print("Hash (contoh)   :", manifest["zips"][0]["sha256"] if manifest["zips"] else "-")

## 4. Catatan & langkah berikutnya

1. **Untuk uji cepat:** set `MAX_DOCS = 1000` agar hanya 1.000 PDF diekstrak (sesuai syarat minimal 1.000 data).
2. **Untuk `full`:** set `MODE = "full"`. Pastikan ruang disk cukup (siapkan **≥ 60 GB**) dan koneksi stabil.
3. Salin **jumlah dokumen**, **tanggal unduh**, dan **hash** ke **Tabel 1.4** di `README.md`.
4. Lanjut ke `notebook/02_indexing.ipynb` (parsing → chunking → embedding → ChromaDB).

> **Verifikasi DOI/lisensi:** lisensi Apache-2.0 tertera di *dataset card*. Dataset berisi
> dokumen hukum publik; tetap patuhi prinsip pelindungan data pribadi saat mengolahnya.